# 03 — Assessment

Compare Phone Recognition output against G2P / CMU targets using edit distance.
Produces per-speaker assessment JSONs and an aggregated `summary.json`.

In [1]:
import sys, json
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))
sys.path.insert(0, str(Path("../../../mod").resolve()))

from config import RESULTS_DIR
from assessment.edit_distance import edit_operations
from powsm.ipa import tokenize_ipa

In [2]:
all_results = []

for pr_file in sorted(RESULTS_DIR.glob("pr/*/*.json")):
    sid = pr_file.parent.name
    speaker = pr_file.stem

    pr_data = json.loads(pr_file.read_text(encoding="utf-8"))
    transcript = pr_data["transcript"]
    actual_phones = pr_data["phone_recognition"]["phones"]

    # Load CMU-based G2P target (one per sentence)
    g2p_file = RESULTS_DIR / "g2p" / sid / "target.json"
    if not g2p_file.exists():
        print(f"WARNING: No G2P target for sentence {sid}, skipping {speaker}")
        continue
    g2p_data = json.loads(g2p_file.read_text(encoding="utf-8"))
    target_phones = g2p_data["g2p"]["phones"]

    # Tokenize PR output for fair comparison (handles list, string, or tuple)
    actual_tokens = tokenize_ipa(actual_phones)

    ops = edit_operations(actual_tokens, target_phones)

    assessment = {
        "sentence_id": sid,
        "speaker": speaker,
        "transcript": transcript,
        "pr_model": pr_data.get("model", "unknown"),
        "g2p_model": g2p_data.get("model", "unknown"),
        "target_phones": target_phones,
        "actual_phones": actual_tokens,
        "ops": ops,
        "total_errors": len(ops),
        "insertions": sum(1 for o in ops if o[0] == "insert"),
        "deletions": sum(1 for o in ops if o[0] == "delete"),
        "substitutions": sum(1 for o in ops if o[0] == "substitute"),
    }

    out_dir = RESULTS_DIR / "assessment" / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f"{speaker}.json").write_text(
        json.dumps(assessment, ensure_ascii=False, indent=2), encoding="utf-8"
    )

    all_results.append(assessment)
    print(f"{speaker} (sentence {sid}): {len(ops)} errors "
          f"(ins={assessment['insertions']}, del={assessment['deletions']}, sub={assessment['substitutions']})")

# Write summary
summary = {
    "total_assessments": len(all_results),
    "avg_errors": sum(r["total_errors"] for r in all_results) / max(len(all_results), 1),
    "results": [
        {"speaker": r["speaker"], "sentence_id": r["sentence_id"], "total_errors": r["total_errors"]}
        for r in all_results
    ],
}
(RESULTS_DIR / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"\nSummary saved. Average errors: {summary['avg_errors']:.1f}")

umit12 (sentence 12): 67 errors (ins=17, del=15, sub=35)
yusuf12 (sentence 12): 66 errors (ins=13, del=12, sub=41)
umit14 (sentence 14): 50 errors (ins=5, del=5, sub=40)

Summary saved. Average errors: 61.0
